In [1]:
import sys
sys.path.append('..')

import copy
import torch
import torch.nn.functional as F
import numpy as np
import pandas as pd
import importlib.util
import matplotlib.pyplot as plt
from tqdm.notebook import tqdm
from torch.utils.data import DataLoader
from pathlib import Path
from sklearn.metrics import accuracy_score, f1_score, mean_absolute_error, confusion_matrix, ConfusionMatrixDisplay

from src.dataset import create_train_val_datasets
from config import TRAIN_SIZE, MAX_K, SPLIT_SEED, MAX_SAMPLES_TRAIN, MAX_SAMPLES_VAL, medium_dataset_path, huge_dataset_path, SPLIT_STRATEGY, SEED_TRAIN, SEED_VAL

In [2]:
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device : {device}") 

learning_strategy = "mse+ce+or"
run_name = Path("runs/run_20260626_141946")
run_path = "../" / run_name / "checkpoints"
model_structure_path = run_path / "model_structure.py"
ckpt_path = run_path / "best_checkpoint.pth"
dataset_path = Path("/sps/l2it/tdonze/gb-dataset-gen/data/synthetic_dataset/dataset_500K_2.0e-23_1.0e-22_filtering_True_10_100_difficulty_1/dataset.hdf5")

Using device : cuda


In [3]:
spec = importlib.util.spec_from_file_location(
    "dynamic_model", model_structure_path
)
dynamic_model_module = importlib.util.module_from_spec(spec)
sys.modules["dynamic_model"] = dynamic_model_module
spec.loader.exec_module(dynamic_model_module)

CardinalityEstimator = getattr(
    dynamic_model_module, "CardinalityEstimator"
)

In [4]:
model = CardinalityEstimator(learning_strategy, max_K=MAX_K, input_channels=4).to(device)
ckpt = torch.load(ckpt_path, map_location=device)
model.load_state_dict(ckpt["model_state_dict"])
model.eval()

CardinalityEstimator(
  (conv_encoder): Sequential(
    (conv_1): Conv1d(4, 32, kernel_size=(7,), stride=(1,), padding=(3,))
    (norm_1): BatchNorm1d(32, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (gelu_1): GELU(approximate='none')
    (conv_2): Conv1d(32, 64, kernel_size=(5,), stride=(2,), padding=(2,))
    (norm_2): BatchNorm1d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (gelu_2): GELU(approximate='none')
    (conv_3): Conv1d(64, 128, kernel_size=(5,), stride=(1,), padding=(2,))
    (norm_3): BatchNorm1d(128, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (gelu_3): GELU(approximate='none')
    (conv_4): Conv1d(128, 196, kernel_size=(3,), stride=(1,), padding=(1,))
    (norm_4): BatchNorm1d(196, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (gelu_4): GELU(approximate='none')
  )
  (pos_encoder): PosEnc()
  (transformer_encoder): TransformerEncoder(
    (layers): ModuleList(
      (0-1): 2 

In [5]:
def make_dataset_view(base_ds, seed, noise):
    ds = copy.copy(base_ds)

    ds.seed = seed
    ds.noise = noise
    ds.rng = np.random.default_rng(seed)
    
    ds.fixed_mixtures = ds._build_fixed_mixtures()

    return ds

_, val_ds, _ = create_train_val_datasets(
    dataset_path=medium_dataset_path,
    train_size=0.0,
    max_K=MAX_K,
    max_samples_val=None,
    noise_val=True,
    deterministic_val=True,
    split_seed=SPLIT_SEED,
    split_strategy=SPLIT_STRATEGY,
    seed_train=SEED_TRAIN,
    seed_val=SEED_VAL,
    return_params=True
)

[09/09/26 16:27:04] INFO     Loading waveforms from                                                                
                             /sps/l2it/tdonze/gb-dataset-gen/data/synthetic_dataset/dataset_500K_uniform_SNR_10_100
                             /dataset.hdf5...

                    INFO     Total number of waveforms in the dataset: 500000

                    INFO     Starting to load waveforms in memory...

[09/09/26 16:27:09] INFO     Waveforms loaded with shape (500000, 4, 128)

                    INFO     Loading parameters in memory...

...

                    INFO     SNR-based split built with 90 bins of width 1: 0 train / 500000 val

                    INFO     Building fixed mixtures for deterministic sampling...

[09/09/26 16:27:13] INFO     Fixed mixtures built successfully.

[09/09/26 16:27:14] INFO     Building fixed mixtures for deterministic sampling...

[09/09/26 16:27:18] INFO     Fixed mixtures built successfully.

In [6]:
results = []
path = Path("evaluation_results_model_triple_head.csv")

signal_ds = copy.copy(val_ds)
signal_ds.noise = False

BATCH_SIZE = 65536

val_loader = DataLoader(val_ds, batch_size=65536, shuffle=False)
signal_loader = DataLoader(signal_ds, batch_size=65536, shuffle=False)

rows = []
offset = 0

with torch.no_grad():
    progress = tqdm(
        zip(val_loader, signal_loader),
        total=len(val_loader),
        desc=f"Evaluating model on validation set",
        leave=True,
    )

    for (summed_waveform, targets, params), (signal_waveform, signal_targets, signal_params) in progress:
        X = torch.as_tensor(summed_waveform, dtype=torch.float32, device=device)
        X_signal = torch.as_tensor(signal_waveform, dtype=torch.float32, device=device)

        targets = torch.as_tensor(targets, dtype=torch.long, device=device)
        signal_targets = torch.as_tensor(signal_targets, dtype=torch.long, device=device)

        assert torch.equal(targets, signal_targets)

        logits = model(X)

        if learning_strategy == "mse":
            y = targets.float().unsqueeze(1) / MAX_K # shape : (batch_size, 1)
            out = torch.sigmoid(logits) # shape : (batch_size, 1)

            pred_score = out.squeeze(1) * MAX_K # shape : (batch_size,)
            pred = torch.round(pred_score).long().clamp(1, MAX_K) # shape : (batch_size,)

            loss_per_sample = F.mse_loss(out, y, reduction="none").squeeze(1) # shape : (batch_size,)
        elif learning_strategy == "ordinal":
            thresholds = torch.arange(1, MAX_K, device=device) # shape3 : (max_K - 1,)
            y = (targets.unsqueeze(1) > thresholds.unsqueeze(0)).float() # shape : (batch_size, max_K - 1)

            probs = torch.sigmoid(logits)

            pred = 1 + (probs > 0.5).sum(dim=1)
            pred = pred.clamp(1, MAX_K)

            pred_score = 1.0 + probs.sum(dim=1)

            loss_per_sample = F.binary_cross_entropy_with_logits(
                logits,
                y,
                reduction="none",
            ).mean(dim=1) 
        elif learning_strategy == "cross_entropy":
            y = targets - 1

            probs = F.softmax(logits, dim=1)
            class_values = torch.arange(1, MAX_K + 1, device=device).float()

            pred = logits.argmax(dim=1) + 1
            pred_score = (probs * class_values.unsqueeze(0)).sum(dim=1)

            loss_per_sample = F.cross_entropy(logits, y, reduction="none")
        if learning_strategy == "mse+ce+or":
            # in that case we process only the logits for the mse head
            logits_mse, logits_ce, logits_or = logits
            y_mse = targets.float().unsqueeze(1) / MAX_K
            out_mse = torch.sigmoid(logits_mse)

            loss_per_sample_mse =  F.mse_loss(out_mse, y_mse, reduction="none").squeeze(1)

            mse_pred_score = out_mse.squeeze(1) * MAX_K
            predicted_mse = torch.round(mse_pred_score).long()
            predicted_mse = predicted_mse.clamp(1, MAX_K)

            out = out_mse
            loss_per_sample = loss_per_sample_mse
            pred_score = mse_pred_score
            pred = predicted_mse

        else:
            raise ValueError(f"Unknown learning strategy : {learning_strategy}")
        processed_logits = out.squeeze() if learning_strategy == "mse" or learning_strategy == "mse+ce+or" else probs # shape : (batch_size,) for mse, (batch_size, max_K - 1) for ordinal, (batch_size, max_K) for cross_entropy
        snr = torch.as_tensor(params["snr"], dtype=torch.float32, device=device)
        source_mask = torch.as_tensor(params["source_mask"], dtype=torch.bool, device=device)

        snr_masked = torch.where(source_mask, snr, torch.zeros_like(snr))
        snr_mix = torch.sqrt((snr_masked ** 2).sum(dim=1))
        max_snr = snr_masked.max(dim=1).values
        mean_snr = snr_masked.sum(dim=1) / source_mask.sum(dim=1).clamp(min=1)

        # n_visible_5 = ((snr > 5) & source_mask).sum(dim=1)
        # n_visible_10 = ((snr > 10) & source_mask).sum(dim=1)
        # n_visible_20 = ((snr > 20) & source_mask).sum(dim=1)

        energy_noisy = (X ** 2).sum(dim=(1, 2))
        energy_signal = (X_signal ** 2).sum(dim=(1, 2))

        batch_size = targets.shape[0]

        source_mask_cpu = source_mask.detach().cpu()
        optional_param_tensors = {
            key: torch.as_tensor(params[key]).detach().cpu()
            for key in [
                "f0",
                "freq_bin_start",
                "freq_bin_stop",
                "freq_bin_peak",
                "freq_support_width",
            ]
            if key in params
        }

        for i in range(batch_size):
            mask_i = source_mask_cpu[i]
            valid_snr = snr[i][source_mask[i]].detach().cpu().numpy()

            row = {
                "run": 0,
                "seed": val_ds.seed,
                "idx": offset + i,

                "true_K": int(targets[i].item()),
                "pred_K": int(pred[i].item()),
                "pred_score": float(pred_score[i].item()),

                "correct": bool(pred[i].item() == targets[i].item()),
                "error_K": int(pred[i].item() - targets[i].item()),
                "abs_error_K": int(abs(pred[i].item() - targets[i].item())),

                "processed_logits": processed_logits[i].item() if learning_strategy == "mse" else processed_logits[i].cpu().numpy().tolist(), # shape : () for mse, (max_K - 1,) for ordinal, (max_K,) for cross_entropy
                "loss": float(loss_per_sample[i].item()),

                "snr_values": valid_snr.tolist(),
                "snr_mix": float(snr_mix[i].item()),
                "max_snr": float(max_snr[i].item()),
                "mean_snr": float(mean_snr[i].item()),

                # "n_visible_5": int(n_visible_5[i].item()), # correspond to the number of sources with SNR > 5
                # "n_visible_10": int(n_visible_10[i].item()), # correspond to the number of sources with SNR > 10
                # "n_visible_20": int(n_visible_20[i].item()), # correspond to the number of sources with SNR > 20

                "energy_noisy": float(energy_noisy[i].item()),
                "energy_signal": float(energy_signal[i].item()),
            }

            if "f0" in optional_param_tensors:
                row["f0_values"] = optional_param_tensors["f0"][i][mask_i].numpy().tolist()
            for result_key in ["freq_bin_start", "freq_bin_stop", "freq_bin_peak", "freq_support_width"]:
                if result_key in optional_param_tensors:
                    row[result_key] = optional_param_tensors[result_key][i][mask_i].numpy().astype(int).tolist()

            rows.append(row)
        offset += batch_size
df = pd.DataFrame(rows)
results.append(df)

Evaluating model on validation set:   0%|          | 0/8 [00:00<?, ?it/s]

In [ ]:
if path.exists():
    print(f"File {path} already exists. Loading existing results.")
    results_df = pd.read_csv(path)
else :
    results_df = pd.concat(results, ignore_index=True)
    print(f"Saving results to {path}")
    results_df.to_csv(path, index=False)

Saving results to evaluation_results_model_triple_head.csv
